# Решения: метрики и валидация

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


## Урок. 1–4. Бинарная задача и четыре числа

In [ ]:
is_eight = (df['label'] == 8).astype(int)
share_8 = float(is_eight.mean())
acc_dumb = float((is_eight == 0).mean())
WHY_ACC_LIES = (
    'Правило «никогда не 8» имеет точность 90% и не находит ни одной восьмёрки: '
    'метрика измеряет частоту класса, а не полезность распознавателя.'
)
X_tr, X_te, y_tr, y_te = train_test_split(df[PIXELS], is_eight, test_size=0.25,
                                          random_state=0, stratify=is_eight)
pred = KNeighborsClassifier(3).fit(X_tr, y_tr).predict(X_te)
tp = int(((pred == 1) & (y_te == 1)).sum())
fp = int(((pred == 1) & (y_te == 0)).sum())
fn = int(((pred == 0) & (y_te == 1)).sum())
tn = int(((pred == 0) & (y_te == 0)).sum())
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1_manual = 2 * precision * recall / (precision + recall)
f1_lib = float(f1_score(y_te, pred))
print(round(share_8, 4), round(acc_dumb, 4), tp, fp, fn, tn,
      round(precision, 4), round(recall, 4), round(f1_manual, 4), round(f1_lib, 4))

## Урок. 5–8. Три части, выбор k, F1 macro/micro

In [ ]:
X_rest, X_final, y_rest, y_final = train_test_split(df[PIXELS], df['label'], test_size=0.2,
                                                    random_state=0, stratify=df['label'])
X_fit, X_val, y_fit, y_val = train_test_split(X_rest, y_rest, test_size=0.25,
                                              random_state=0, stratify=y_rest)
rows = []
for k in (1, 3, 5, 7, 9, 15):
    m = KNeighborsClassifier(k).fit(X_fit, y_fit)
    rows.append([k, float(accuracy_score(y_val, m.predict(X_val)))])
val_table = pd.DataFrame(rows, columns=['k', 'accuracy'])
best_k = int(val_table.sort_values('accuracy', ascending=False).iloc[0]['k'])
final_model = KNeighborsClassifier(best_k).fit(X_fit, y_fit)
pred_final = final_model.predict(X_final)
acc_final = float(accuracy_score(y_final, pred_final))
WHY_ONE_SHOT = (
    'Финальная часть заменяет будущую почту. Каждый повторный взгляд превращает её в ещё одну '
    'проверочную: мы начинаем подбирать настройку под неё, и оценка становится завышенной.'
)
f1_micro = float(f1_score(y_final, pred_final, average='micro'))
F1_LIMIT = (
    'В сбалансированной задаче про десять цифр micro-F1 совпал с точностью: '
    'F1 нужен там, где один класс редкий и цена пропуска высока.'
)
f1_macro = float(f1_score(y_final, pred_final, average='macro'))
MACRO_NOTE = (
    'Macro усредняет по цифрам без учёта их частоты: если одна редкая цифра распознаётся плохо, '
    'macro просядет, а micro почти не заметит.'
)
print(val_table, best_k, round(acc_final, 4), round(f1_micro, 4), round(f1_macro, 4))

## ДЗ. 1–4

In [ ]:
is_three = (df['label'] == 3).astype(int)
A_tr, A_te, b_tr, b_te = train_test_split(df[PIXELS], is_three, test_size=0.25,
                                          random_state=0, stratify=is_three)
p3 = KNeighborsClassifier(3).fit(A_tr, b_tr).predict(A_te)
tp3 = int(((p3 == 1) & (b_te == 1)).sum()); fp3 = int(((p3 == 1) & (b_te == 0)).sum())
fn3 = int(((p3 == 0) & (b_te == 1)).sum())
precision_3 = tp3 / (tp3 + fp3)
recall_3 = tp3 / (tp3 + fn3)
f1_3 = float(f1_score(b_te, p3))
C_tr, C_te, d_tr, d_te = train_test_split(df[PIXELS], df['label'], test_size=0.25,
                                          random_state=0, stratify=df['label'])
pm = KNeighborsClassifier(3).fit(C_tr, d_tr).predict(C_te)
cm = confusion_matrix(d_te, pm)
errors_per_digit = cm.sum(axis=1) - cm.diagonal()
worst_digit = int(errors_per_digit.argmax())
is8 = (df['label'] == 8).astype(int)
E_rest, E_final, f_rest, f_final = train_test_split(df[PIXELS], is8, test_size=0.2,
                                                    random_state=0, stratify=is8)
E_fit, E_val, f_fit, f_val = train_test_split(E_rest, f_rest, test_size=0.25,
                                              random_state=0, stratify=f_rest)
scores = []
for k in (1, 3, 5, 7, 9):
    m = KNeighborsClassifier(k).fit(E_fit, f_fit)
    scores.append([k, float(f1_score(f_val, m.predict(E_val)))])
best_k_bin = int(pd.DataFrame(scores, columns=['k', 'f1'])
                 .sort_values('f1', ascending=False).iloc[0]['k'])
f1_final = float(f1_score(f_final, KNeighborsClassifier(best_k_bin)
                          .fit(E_fit, f_fit).predict(E_final)))
PROTOCOL = (
    'Делю таблицу на три части со stratify: 60% обучение, 20% проверка, 20% финал. '
    'Все настройки — число соседей, масштабирование, набор признаков — выбираю по проверочной части. '
    'Масштаб считаю только по обучающей. Финальную часть смотрю один раз в конце и записываю '
    'полученное число в отчёт вместе с baseline самой частой цифры.'
)
print(round(f1_3, 4), worst_digit, best_k_bin, round(f1_final, 4))